In [1]:
import pandas as pd
import json
import numpy as np

In [2]:
path = "E:\\DATASCIENCE PORTFOLIO\\Practice_Projects\\Instacart Data Transformation\\Dataset\\"

df = pd.read_json(f'{path}case.json')

In [3]:
df.head()

,EnqueuedTimeUtc,EventName,Payload
0,2021-09-05 08:04:08 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
1,2021-08-18 11:43:23 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPricePerOption"",""offe..."
2,2021-09-05 09:04:04 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
3,2021-08-25 05:02:55 UTC,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
4,2021-09-05 08:03:28 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   EnqueuedTimeUtc  37 non-null     object
 1   EventName        37 non-null     object
 2   Payload          37 non-null     object
dtypes: object(3)
memory usage: 1020.0+ bytes


In [5]:
brazil_tz = pd.to_datetime(df['EnqueuedTimeUtc']).dt.tz_convert('America/Sao_Paulo')
df['EnqueuedTimeUtc'] = brazil_tz.dt.strftime('%d/%m/%y')

In [6]:
curate_offer = df[df['EventName'] == 'CurateOffer_Result'].reset_index(drop = True)
curate_offer.head()

,EnqueuedTimeUtc,EventName,Payload
0,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
1,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""c99a..."
2,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""d668..."
3,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""0a06..."
4,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""135d..."


In [7]:
co_list = []
for idx, row in curate_offer.iterrows():
    data = json.loads(row['Payload'])
    payload_data = pd.json_normalize(data,
                            record_path = ['options'],
                            meta = ['curationProvider', 'offerId', 'dealerId'])
    payload_data['EnqueuedTimeSP'] = row['EnqueuedTimeUtc']

    co_list.append(payload_data)
    

CuratedOfferOptions = pd.concat(co_list, axis= 0).reset_index(drop = True)


column_order = {'curationProvider':'CurationProvider', 
                'offerId':'OfferId', 
                'dealerId':'DealerId', 
                'uniqueOptionId':'UniqueOptionId', 
                'optionId':'OptionId',
                'isMobileDealer':'IsMobileDealer', 
                'isOpen':'IsOpen', 
                'eta':'Eta',
                'chamaScore':'ChamaScore',
                'productBrand':'ProductBrand',
                'isWinner':'IsWinner', 
                'minimumPrice':'MinimumPrice',
                'maximumPrice':'MaximumPrice', 
                'dynamicPrice':'DynamicPrice', 
                'finalPrice':'FinalPrice', 
                'defeatPrimaryReason':'DefeatPrimaryReason', 
                'defeatReasons':'DefeatReasons', 
                'EnqueuedTimeSP':'EnqueuedTimeSP'}

CuratedOfferOptions = CuratedOfferOptions[column_order.keys()].rename(columns = column_order)
CuratedOfferOptions = CuratedOfferOptions.fillna('')
ob_col = CuratedOfferOptions.select_dtypes(include=['object']).columns
CuratedOfferOptions[ob_col] = CuratedOfferOptions[ob_col].astype(str)


In [20]:
CuratedOfferOptions

,CurationProvider,OfferId,DealerId,UniqueOptionId,OptionId,IsMobileDealer,IsOpen,Eta,ChamaScore,ProductBrand,IsWinner,MinimumPrice,MaximumPrice,DynamicPrice,FinalPrice,DefeatPrimaryReason,DefeatReasons,EnqueuedTimeSP
0,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,b0e296a9-0590-f0e0-8211-243a2ededb12,6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2,True,True,1:00,8.0,ULTRAGAZ,True,90.00,180.00,91.90,91.90,,,25/08/21
1,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,d6562c24-0b37-5fb4-8275-65b7b8b47b87,6517 || 6517,False,False,0:01,8.0,ULTRAGAZ,False,90.00,180.00,91.90,91.90,Closed,"['Closed', 'HasDriverInOffer']",25/08/21
2,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,8d0f9262-f543-d0c8-a869-33985ae3ecda,9047 || 9047 || ULTRAGAZ,False,False,1:00,9.0,ULTRAGAZ,False,99.00,198.00,99.95,99.95,Closed,"['Closed', 'HigherPrice', 'HasDriverInOffer']",25/08/21
3,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,9047 || 9047 || CONSIGAZ,False,False,1:00,9.0,CONSIGAZ,False,89.99,179.98,91.89,91.89,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/21
4,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,577e4bbd-f49d-ac23-56a6-e70072a05229,9047 || 9047 || LIQUIGAS,False,False,1:00,9.0,LIQUIGAS,False,92.00,184.00,93.90,93.90,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/21
5,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,f9b876ab-2590-952f-d69d-5b352ec251f3,9047 || 9047 || NACIONALGAS,False,False,1:00,9.0,NACIONALGAS,True,88.90,177.80,90.80,90.80,,,25/08/21
6,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,8330,151e59ac-761a-96f5-d2b9-882037a9fd28,8330 || 8330 || CONSIGAZ,False,True,0:01,8.0,CONSIGAZ,True,80.00,160.00,87.35,87.35,,,25/08/21
7,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,8835,b7a7b6d1-4dae-7392-5aaf-f3369c29db1d,8835 || 8835 || LIQUIGAS,False,False,0:01,8.0,LIQUIGAS,True,60.00,120.00,87.35,87.35,,,25/08/21
8,ByPrice,c99aa9a7-ac50-4a10-be0f-ac9f5ac0f45d,6517,b0e296a9-0590-f0e0-8211-243a2ededb12,6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2,True,True,00:12:54.9215999,8.0,ULTRAGAZ,True,90.00,180.00,91.90,91.90,,,25/08/21
9,ByPrice,c99aa9a7-ac50-4a10-be0f-ac9f5ac0f45d,6517,d6562c24-0b37-5fb4-8275-65b7b8b47b87,6517 || 6517,False,True,0:01,8.0,ULTRAGAZ,False,90.00,180.00,91.90,91.90,HasDriverInOffer,['HasDriverInOffer'],25/08/21


In [9]:
df.head()

,EnqueuedTimeUtc,EventName,Payload
0,05/09/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
1,18/08/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPricePerOption"",""offe..."
2,05/09/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
3,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
4,05/09/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."


In [10]:
dynamic_price = df[df['EventName'] == 'DynamicPrice_Result'].reset_index(drop = True)
dynamic_price['EnqueuedTimeUtc'].value_counts()

EnqueuedTimeUtc
05/09/21    25
18/08/21     7
Name: count, dtype: int64

In [11]:
dpoption_list = []
dprrange_list = []
for idx, row in dynamic_price.iterrows():
    
    data = json.loads(row['Payload'])
    
    if data['provider'] == 'ApplyDynamicPriceRange':
        
        dprange = pd.json_normalize(data)
        dprange['EnqueuedTimeSP'] = row['EnqueuedTimeUtc']
        dprrange_list.append(dprange)
        
    if data['provider'] == 'ApplyDynamicPricePerOption':

        dpoption = pd.json_normalize(data,
                                    record_path=['algorithmOutput'],
                                    meta = ['provider','offerId'])
        dpoption['EnqueuedTimeSP'] = row['EnqueuedTimeUtc']
        dpoption_list.append(dpoption)

    else:
        pass



DynamicPriceOption = pd.concat(dpoption_list, axis = 0, ignore_index = True)
DynamicPriceRange = pd.concat(dprrange_list,axis = 0, ignore_index= True)


dpoption_col = {'provider':'Provider', 
                'offerId':'OfferId', 
                'uniqueOptionId':'UniqueOptionId', 
                'bestPrice':'BestPrice', 
                'EnqueuedTimeSP':'EnqueuedTimeSP'}

DynamicPriceOption = DynamicPriceOption[dpoption_col.keys()].rename(columns = dpoption_col)
DynamicPriceOption.columns



dprange_col = {'provider':'Provider', 
               'offerId':'OfferId', 
                'algorithmOutput.min_global':'MinimumGlobal',
               'algorithmOutput.min_recommended':'MaximumRecommended', 
               'algorithmOutput.max_recommended' : 'MaximumRecommended',
               'algorithmOutput.differenceMinRecommendMinTheory' : 'DifferenceMinRecommendMinTheory', 
               'EnqueuedTimeSP':'EnqueuedTimeSP'}

DynamicPriceRange = DynamicPriceRange[dprange_col.keys()].rename(columns = dprange_col)





In [12]:
DynamicPriceRange.head()

,Provider,OfferId,MinimumGlobal,MaximumRecommended,MaximumRecommended,DifferenceMinRecommendMinTheory,EnqueuedTimeSP
0,ApplyDynamicPriceRange,a6611d55-9624-4381-8cdd-323ee3689241,85.00,87.20,97.65,2.2,05/09/21
1,ApplyDynamicPriceRange,b8c636fa-8241-47dc-ac40-bdf438a04d9c,85.00,87.20,97.65,2.2,05/09/21
2,ApplyDynamicPriceRange,3d32f7fb-396d-4d3f-b673-dea1f7dc41b7,85.00,87.20,97.65,2.2,05/09/21
3,ApplyDynamicPriceRange,329194f3-95a4-45ef-b3d0-2796f74ce2a0,85.00,87.20,97.65,2.2,05/09/21
4,ApplyDynamicPriceRange,fdcfde5c-113d-4a59-9ae0-8bc31e2943d8,87.35,89.25,99.95,1.9,05/09/21


In [13]:
DynamicPriceOption.head()

,Provider,OfferId,UniqueOptionId,BestPrice,EnqueuedTimeSP
0,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,b0e296a9-0590-f0e0-8211-243a2ededb12,92.45,18/08/21
1,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,d6562c24-0b37-5fb4-8275-65b7b8b47b87,92.45,18/08/21
2,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,8d0f9262-f543-d0c8-a869-33985ae3ecda,92.45,18/08/21
3,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,151e59ac-761a-96f5-d2b9-882037a9fd28,94.60,18/08/21
4,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,94.60,18/08/21


In [21]:
CuratedOfferOptions.to_csv(f'{path}CuratedOfferOptions.csv',index=False)
DynamicPriceOption.to_csv(f'{path}DynamicPriceOption.csv',index = False)
DynamicPriceRange.to_csv(f'{path}DynamicPriceRange.csv', index = False)

,CurationProvider,OfferId,DealerId,UniqueOptionId,OptionId,IsMobileDealer,IsOpen,Eta,ChamaScore,ProductBrand,IsWinner,MinimumPrice,MaximumPrice,DynamicPrice,FinalPrice,DefeatPrimaryReason,DefeatReasons,EnqueuedTimeSP
0,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,b0e296a9-0590-f0e0-8211-243a2ededb12,6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2,True,True,1:00,8.0,ULTRAGAZ,True,90.00,180.00,91.90,91.90,NaN,NaN,25/08/21
1,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,d6562c24-0b37-5fb4-8275-65b7b8b47b87,6517 || 6517,False,False,0:01,8.0,ULTRAGAZ,False,90.00,180.00,91.90,91.90,Closed,"['Closed', 'HasDriverInOffer']",25/08/21
2,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,8d0f9262-f543-d0c8-a869-33985ae3ecda,9047 || 9047 || ULTRAGAZ,False,False,1:00,9.0,ULTRAGAZ,False,99.00,198.00,99.95,99.95,Closed,"['Closed', 'HigherPrice', 'HasDriverInOffer']",25/08/21
3,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,9047 || 9047 || CONSIGAZ,False,False,1:00,9.0,CONSIGAZ,False,89.99,179.98,91.89,91.89,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/21
4,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,577e4bbd-f49d-ac23-56a6-e70072a05229,9047 || 9047 || LIQUIGAS,False,False,1:00,9.0,LIQUIGAS,False,92.00,184.00,93.90,93.90,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/21
